# NUFROST Colab Launcher

> This is a launch script to load and run the NUFROST reconstruction algorithm in Google Colab environment.
You can open this notebook in Google Colab by right-clicking in Google Drive and selecting "Open with > Google Colaboratory".

## 1. Configuration Section
> configurable parameters for the NUFROST Colab Launcher.

In [ ]:
from pathlib import Path
from tkinter.tix import IMAGE

GOOGLE_DRIVE_MOUNT_POINT = Path("/content/drive")
GOOGLE_DRIVE_ROOT_PATH = GOOGLE_DRIVE_MOUNT_POINT / "MyDrive"
PROJECT_PATH = GOOGLE_DRIVE_ROOT_PATH / "WorkSpaces" / "nufrost"

IMAGE_FILENAMES: list[str] = None
TARGET_LON: float = 120.0
TARGET_LAT: float = 30.0
TARGET_BAND: str = "BLUE"
CACHE_DIR_IN_COLAB = PROJECT_PATH / "data" / "cache" / "colab"
TARGET_TIME: str = "2023-06-15 12:00:00"


## 2. Mount Google Drive

In [ ]:
from google.colab import drive # type: ignore[import]
import os
import sys

# mount Google Drive to colab
drive.mount(str(GOOGLE_DRIVE_MOUNT_POINT))

# Check if the specified project root exists
if not PROJECT_PATH.exists():
    print(f"[Warning] Path not found: {PROJECT_PATH}")
    print("Please make sure you have uploaded the code and modified the PROJECT_PATH variable above.")
else:
    # Add the project root directory to Python search path to enable importing src
    if PROJECT_PATH not in sys.path:
        sys.path.append(str(PROJECT_PATH))
    # Change working directory to the project root
    os.chdir(str(PROJECT_PATH))
    print(f"[Success] Working directory changed to: {os.getcwd()}")

## 3. Install Dependencies
use `requirements.txt` to install necessary packages.

In [ ]:
%pip install -r requirements.txt

## 4. Run Reconstruction Task

Here we directly call the `src.reconstruct` interface for reconstruction.

In [ ]:
import src.data_loader
from src.data_loader import find_image_chunks
import os
import src
import importlib
from pathlib import Path

importlib.reload(src) # Ensure latest code is loaded

IMAGE_FOLDER = PROJECT_PATH / "data" / "hls"
OUTPUT_DIR_PATH = PROJECT_PATH / "data" / "output" / "hants_out"
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

image

if IMAGE_FILENAMES:
    image_paths_list = [[str(IMAGE_FOLDER / image_filename)] for image_filename in IMAGE_FILENAMES]
else:
    image_paths = find_image_chunks(
        data_dir=IMAGE_FOLDER,
        lon=TARGET_LON,
        lat=TARGET_LAT,
        band=TARGET_BAND,
        cache_dir=CACHE_DIR_IN_COLAB
    )
    image_paths_list = [image_paths] if image_paths else []

for image_paths in image_paths_list:
    first_path = Path(image_paths[0])
    OUTPUT_PATH = str(OUTPUT_DIR_PATH / f"{first_path.name}_{TARGET_TIME}_recon_result.tif")

    if Path(OUTPUT_PATH).exists():
        print(f"Skipping {first_path.name}, output already exists.")
        continue

    src.reconstruct_hants(
        image=image_paths,
        target_time=TARGET_TIME,
        output_path=OUTPUT_PATH,
        n_jobs=N_JOBS,
        cache_dir=CACHE_DIR_IN_COLAB
    )

importlib.reload(src.data_loader)
